In [ ]:
#### filter invalid sample: DeQA(fake) > DeQA(real) ####
import os
import pandas as pd

ext_list = [".png", ".PNG", ".jpg", ".jpeg", ".JPG", ".JPEG"]
real_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/real"
fake_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/fake"

all_information_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train.csv"
fake_deqa_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/deqa_fake_score.csv"
real_deqa_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/deqa_real_score.csv"
output_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/train.csv"

# Load dataframes
all_information_df = pd.read_csv(all_information_csv_file_path)
fake_deqa_df = pd.read_csv(fake_deqa_csv_file_path)
real_deqa_df = pd.read_csv(real_deqa_csv_file_path)

# Check column names (in case deqa column name varies)
print("fake_deqa_df columns:", fake_deqa_df.columns.tolist())
print("real_deqa_df columns:", real_deqa_df.columns.tolist())
print("all_information_df columns:", all_information_df.columns.tolist())

fake_deqa_col, real_deqa_col = "deqa_score", "deqa_score"

# Step 1: Find common UIDs
common_uid_list = list(set(fake_deqa_df['uid']) & set(real_deqa_df['uid']))
print(f"\nStep 1: Found {len(common_uid_list)} common UIDs")

# Step 2: Filter UIDs where real_deqa > fake_deqa
# Create lookup dictionaries for faster access
fake_deqa_dict = dict(zip(fake_deqa_df['uid'], fake_deqa_df[fake_deqa_col]))
real_deqa_dict = dict(zip(real_deqa_df['uid'], real_deqa_df[real_deqa_col]))

valid_uid_list = []
for uid in common_uid_list:
    real_score = real_deqa_dict[uid]
    fake_score = fake_deqa_dict[uid]
    # if real_score > fake_score: # add_noise-denoise-half
    if real_score > fake_score and real_score > 3.5:
        win_image_path = None
        for ext in ext_list:
            win_image_path = os.path.join(real_image_dir, f"{uid}{ext}")
            if os.path.exists(win_image_path):
                break
        if win_image_path is None:
            print(f"Warning: No valid image found for uid {uid}")
            continue
        
        lose_image_path = None
        for ext in ext_list:
            lose_image_path = os.path.join(fake_image_dir, f"{uid}{ext}")
            if os.path.exists(lose_image_path):
                break
        if lose_image_path is None:
            print(f"Warning: No valid image found for uid {uid}")
        
        prompt = all_information_df.loc[all_information_df['uid'] == uid, 'PROMPT'].values[0]
        
        valid_uid_list.append({
            'uid': uid,
            'prompt': prompt,
            'win_image_path': win_image_path,
            'lose_image_path': lose_image_path
        })

print(f"Step 3: Writing {len(valid_uid_list)} valid samples to {output_csv_file_path}")
pd.DataFrame(valid_uid_list).to_csv(output_csv_file_path, index=False)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import random

csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/random_add_noise_step/train.csv"
df = pd.read_csv(csv_file_path)

row = random.randint(0, len(df)-1)

win_image_path = df.iloc[row]['win_image_path']
lose_image_path = df.iloc[row]['lose_image_path']
uid = df.iloc[row]['uid']

win_image = Image.open(win_image_path)
lose_image = Image.open(lose_image_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)

axes[0].imshow(lose_image)
axes[0].set_title("Generated Image")
axes[0].axis('off') 

axes[1].imshow(win_image)
axes[1].set_title("Real Image")
axes[1].axis('off')

fig.suptitle(f"UID: {uid}", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
